# Live 스트리밍 테스트용 영상 전처리 노트북

이 노트북은 RTSP/NVR에서 받은 원본 영상을 **실제 live 스트리밍 파이프라인 입력 조건에 가깝게** 변환하기 위한 전용 노트북입니다.

기존 `video_preprocess_demo.ipynb`는 OpenCV `VideoWriter` 기반의 범용 전처리 예제입니다. 반복 테스트용 입력 영상을 만들 때는 OpenCV `mp4v`보다 `ffmpeg + libx264(H.264)`를 쓰는 편이 실제 RTSP/NVR 입력 조건에 더 가깝고, 용량 제어도 명확합니다.

## 이 노트북의 목표

1. 원본 CCTV/NVR mp4를 읽습니다.
2. live 파이프라인 입력과 같은 `1280x720`, `10fps`로 맞춥니다.
3. H.264(`libx264`)로 다시 저장합니다.
4. 변환 전/후 `codec`, `fps`, `해상도`, `bitrate`, `용량`을 비교합니다.
5. 샘플 프레임을 추출해서 눈으로 확인할 수 있게 합니다.

## 중요한 전제

- 여기서 만드는 파일은 서비스에 바로 투입하는 최종 산출물이 아니라, **offline 반복 테스트용 입력 영상**입니다.
- live 파이프라인이 실제로는 RTSP stream을 decode해서 frame을 뽑는다면, 이 노트북은 그 입력 조건을 파일 기반으로 최대한 비슷하게 재현합니다.
- 용량은 `fps`와 해상도뿐 아니라 `codec`, `CRF`, `GOP`, `B-frame` 설정에 크게 좌우됩니다.
- 원본 NVR 영상이 이미 강하게 압축된 H.264라면, 10fps로 낮춰도 설정에 따라 용량이 커질 수 있습니다.


## 1. 프로젝트 루트와 기본 경로 설정

아래 셀에서 가장 먼저 바꿀 값은 `INPUT_ROOT`입니다.

- `INPUT_ROOT`: 원본 영상들이 들어있는 폴더 또는 단일 영상 파일
- `OUTPUT_ROOT`: 변환 결과를 저장할 폴더
- `SCAN_DEPTH`: 폴더 입력일 때 몇 단계까지 탐색할지 결정합니다.

처음에는 `SCAN_DEPTH = 0`으로 두는 것을 추천합니다. 입력 폴더 바로 아래 mp4만 처리하므로 실수로 하위 폴더 전체를 오래 변환하는 일을 줄일 수 있습니다.


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
from datetime import datetime
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """notebooks 폴더에서 실행해도 repo root를 찾기 위한 helper입니다."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("project root not found")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# TODO: 필요하면 이 경로만 바꿔서 사용하세요.
INPUT_ROOT = Path("/share_ssd/ltb/Users/ltb/박스_추론용_샘플영상들/260423_서초_서리풀_원본_영상들/260507_샘플다시저장_원본영상")

# 변환 결과는 repo 내부 artifacts 아래에 저장합니다. raw 원본은 건드리지 않습니다.
RUN_NAME = datetime.now().strftime("%Y%m%d_%H%M%S_260507_sample_live_720p_10fps_aligned")
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME

# 0이면 INPUT_ROOT 바로 아래 파일만 처리합니다. None이면 재귀 탐색합니다.
SCAN_DEPTH = 0

print("PROJECT_ROOT =", PROJECT_ROOT)
print("INPUT_ROOT   =", INPUT_ROOT)
print("OUTPUT_ROOT  =", OUTPUT_ROOT)


## 2. Live 테스트용 인코딩 preset 설정

서비스 입력이 `720p`, `10fps`라고 가정하고 아래 값을 기본값으로 둡니다.

### CRF 선택 기준

- `CRF=20~22`: 품질 우선, 용량 큼
- `CRF=23`: 일반적인 균형값
- `CRF=26~28`: 반복 테스트용으로 더 가벼움, 품질 손실 증가

### LOW_LATENCY_MODE

`LOW_LATENCY_MODE=True`이면 `-tune zerolatency`, `-bf 0`을 사용합니다. 실제 live stream에 더 가까운 GOP 구조를 만들 수 있지만, 용량은 조금 커질 수 있습니다.

반복 테스트용 파일 크기가 더 중요하면 `False`로 바꿔도 됩니다.


In [ ]:
TARGET_WIDTH = 1280
TARGET_HEIGHT = 720
TARGET_FPS = 10

# multi-CCTV 테스트용으로 3개 출력 영상의 길이와 frame count를 동일하게 맞춥니다.
# None이면 입력 영상 중 가장 짧은 duration을 TARGET_FPS 단위로 내림해서 자동 계산합니다.
COMMON_DURATION_SEC = None
ALIGN_TO_SHORTEST_VIDEO = True

# live 유사성 우선: 20 = 10fps 기준 약 2초마다 keyframe
GOP = TARGET_FPS * 2

# 반복 테스트용 기본값입니다. 용량이 크면 26 또는 28로 올려보세요.
CRF = 23

# veryfast는 속도/용량 균형이 좋습니다. 용량을 더 줄이고 시간이 괜찮으면 medium도 가능합니다.
PRESET = "veryfast"

# 실제 RTSP low-latency 성격을 더 가깝게 흉내내고 싶으면 True를 유지하세요.
LOW_LATENCY_MODE = True

# 오디오는 모델 입력에 필요 없으므로 제거합니다.
DROP_AUDIO = True

# 카메라별 추가 보정 filter입니다.
# 파일명에 '#1'이 포함된 영상은 상하좌우가 뒤집힌 상태로 보고 hflip,vflip을 먼저 적용합니다.
# 다른 카메라도 방향/crop 보정이 필요하면 '#2', '#3' 값을 추가하거나 수정하세요.
CAMERA_FILTERS = {
    "#1": ["hflip", "vflip"],
    "#2": [],
    "#3": [],
}

print({
    "TARGET_WIDTH": TARGET_WIDTH,
    "TARGET_HEIGHT": TARGET_HEIGHT,
    "TARGET_FPS": TARGET_FPS,
    "COMMON_DURATION_SEC": COMMON_DURATION_SEC,
    "ALIGN_TO_SHORTEST_VIDEO": ALIGN_TO_SHORTEST_VIDEO,
    "GOP": GOP,
    "CRF": CRF,
    "PRESET": PRESET,
    "LOW_LATENCY_MODE": LOW_LATENCY_MODE,
    "DROP_AUDIO": DROP_AUDIO,
    "CAMERA_FILTERS": CAMERA_FILTERS,
})


## 3. 입력 영상 목록 만들기

아래 셀은 처리할 영상 목록을 출력합니다.

처음 실행할 때는 `MAX_VIDEOS_FOR_TEST = 1`로 두고 영상 1개만 변환해보는 것을 추천합니다. 결과가 마음에 들면 `None`으로 바꿔 전체를 처리하세요.


In [ ]:
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi"}

# 처음에는 1개만 테스트하세요. 전체 처리하려면 None으로 바꾸세요.
MAX_VIDEOS_FOR_TEST = None


def collect_videos(input_root: Path, scan_depth: int | None = 0) -> list[Path]:
    input_root = input_root.expanduser().resolve()
    if input_root.is_file():
        return [input_root] if input_root.suffix.lower() in VIDEO_EXTS else []
    if not input_root.exists():
        raise FileNotFoundError(input_root)
    if scan_depth is None:
        files = [p for p in input_root.rglob("*") if p.suffix.lower() in VIDEO_EXTS]
    else:
        files = []
        for p in input_root.glob("*"):
            if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
                files.append(p)
    return sorted(files)


videos = collect_videos(INPUT_ROOT, SCAN_DEPTH)
if MAX_VIDEOS_FOR_TEST is not None:
    videos = videos[:MAX_VIDEOS_FOR_TEST]

print(f"처리 대상 영상 수: {len(videos)}")
for idx, path in enumerate(videos):
    print(f"[{idx:03d}] {path}")


## 4. 원본 영상 정보 확인

`ffprobe`로 원본의 codec, 해상도, fps, bitrate, duration, size를 확인합니다.

여기서 원본이 이미 `h264`, `1280x720`, 낮은 bitrate라면, 단순히 fps를 낮췄다고 항상 용량이 줄지는 않습니다. 최종 용량은 `CRF`, `GOP`, `B-frame`, 움직임 복잡도에 영향을 받습니다.


In [ ]:
def run_cmd(cmd: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, check=True, capture_output=True, text=True)


def ffprobe_json(path: Path) -> dict:
    cmd = [
        "ffprobe", "-v", "error",
        "-print_format", "json",
        "-show_format", "-show_streams",
        str(path),
    ]
    return json.loads(run_cmd(cmd).stdout)


def parse_rate(value: str | None) -> float:
    if not value or value == "0/0":
        return 0.0
    if "/" in value:
        n, d = value.split("/", 1)
        return float(n) / float(d) if float(d) else 0.0
    return float(value)


def video_summary(path: Path) -> dict:
    data = ffprobe_json(path)
    stream = next(s for s in data["streams"] if s.get("codec_type") == "video")
    fmt = data["format"]
    size_bytes = int(fmt.get("size", 0) or 0)
    duration = float(stream.get("duration") or fmt.get("duration") or 0.0)
    bitrate = float(stream.get("bit_rate") or fmt.get("bit_rate") or 0.0)
    return {
        "path": str(path),
        "name": path.name,
        "codec": stream.get("codec_name"),
        "width": int(stream.get("width") or 0),
        "height": int(stream.get("height") or 0),
        "fps": round(parse_rate(stream.get("avg_frame_rate")), 3),
        "duration_sec": round(duration, 3),
        "bitrate_kbps": round(bitrate / 1000, 1) if bitrate else 0.0,
        "size_mb": round(size_bytes / 1024 / 1024, 2),
    }


source_summaries = [video_summary(p) for p in videos]
source_summaries


## 5. ffmpeg 명령 생성

아래 셀은 실제 변환 명령을 생성합니다.

파일명에 `#1`이 들어간 영상은 `CAMERA_FILTERS` 설정에 따라 `hflip,vflip`이 먼저 적용됩니다. 즉, cam1 영상은 방향을 바로잡은 뒤 10fps/720p 변환을 수행합니다.

핵심 filter는 다음과 같습니다.

```text
# cam1 예시
hflip,vflip,
fps=10,
scale=1280:720:force_original_aspect_ratio=decrease,
pad=1280:720:(ow-iw)/2:(oh-ih)/2,
setsar=1
```

의미:

- `fps=10`: 시간 기준으로 10fps 샘플링
- `scale=...decrease`: 원본 비율을 유지하면서 1280x720 안에 들어오게 resize
- `pad`: 남는 영역을 padding해서 최종 frame size를 1280x720으로 고정
- `setsar=1`: pixel aspect ratio를 일반적인 정사각 픽셀로 고정


In [ ]:
def camera_filters_for(source_path: Path) -> list[str]:
    """파일명에 들어있는 '#1', '#2', '#3' 표기를 기준으로 카메라별 filter를 반환합니다."""
    name = source_path.name
    for camera_key, filters in CAMERA_FILTERS.items():
        if camera_key in name:
            return list(filters)
    return []


def camera_filter_tag(source_path: Path) -> str:
    return "_camfix" if camera_filters_for(source_path) else ""


def output_path_for(source_path: Path, output_root: Path) -> Path:
    rel = source_path.name if INPUT_ROOT.is_file() else source_path.relative_to(INPUT_ROOT).as_posix()
    rel_path = Path(rel)
    stem = rel_path.stem + f"__live{camera_filter_tag(source_path)}_{TARGET_WIDTH}x{TARGET_HEIGHT}_{TARGET_FPS}fps_crf{CRF}"
    return output_root / rel_path.parent / f"{stem}.mp4"


def common_output_duration_sec(summaries: list[dict]) -> float | None:
    """공통 출력 길이를 10fps frame boundary에 맞춰 계산합니다."""
    if COMMON_DURATION_SEC is not None:
        return float(COMMON_DURATION_SEC)
    if not ALIGN_TO_SHORTEST_VIDEO:
        return None
    min_duration = min(row["duration_sec"] for row in summaries)
    common_frames = int(min_duration * TARGET_FPS)
    return common_frames / TARGET_FPS


COMMON_OUTPUT_DURATION_SEC = common_output_duration_sec(source_summaries)
COMMON_OUTPUT_FRAMES = None if COMMON_OUTPUT_DURATION_SEC is None else int(COMMON_OUTPUT_DURATION_SEC * TARGET_FPS)

print("COMMON_OUTPUT_DURATION_SEC =", COMMON_OUTPUT_DURATION_SEC)
print("COMMON_OUTPUT_FRAMES =", COMMON_OUTPUT_FRAMES)


def build_ffmpeg_cmd(source_path: Path, output_path: Path) -> list[str]:
    filters = [
        *camera_filters_for(source_path),
        f"fps={TARGET_FPS}",
        f"scale={TARGET_WIDTH}:{TARGET_HEIGHT}:force_original_aspect_ratio=decrease",
        f"pad={TARGET_WIDTH}:{TARGET_HEIGHT}:(ow-iw)/2:(oh-ih)/2",
        "setsar=1",
    ]
    vf = ",".join(filters)
    cmd = [
        "ffmpeg", "-y",
        "-i", str(source_path),
        "-map", "0:v:0",
    ]
    if COMMON_OUTPUT_DURATION_SEC is not None:
        cmd += ["-t", str(COMMON_OUTPUT_DURATION_SEC)]
    if DROP_AUDIO:
        cmd += ["-an"]
    cmd += [
        "-sn", "-dn",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        "-g", str(GOP),
        "-keyint_min", str(GOP),
        "-sc_threshold", "0",
        "-movflags", "+faststart",
    ]
    if COMMON_OUTPUT_FRAMES is not None:
        cmd += ["-frames:v", str(COMMON_OUTPUT_FRAMES)]
    if LOW_LATENCY_MODE:
        cmd += ["-tune", "zerolatency", "-bf", "0"]
    cmd += [str(output_path)]
    return cmd


planned = []
for source in videos:
    out_path = output_path_for(source, OUTPUT_ROOT)
    planned.append((source, out_path, build_ffmpeg_cmd(source, out_path)))

for source, out_path, cmd in planned:
    print("SOURCE:", source)
    print("OUTPUT:", out_path)
    print("CMD:", " ".join(shlex.quote(x) for x in cmd))
    print()


## 6. 변환 실행

처음에는 영상 1개만 변환한 뒤 결과를 확인하세요.

전체 변환은 시간이 걸릴 수 있습니다. 특히 원본이 1시간짜리 여러 개라면 CPU 인코딩 시간이 꽤 걸릴 수 있습니다.


In [ ]:
RUN_TRANSCODE = True  # Run All 하면 실제 변환까지 진행됩니다.

if not RUN_TRANSCODE:
    print("RUN_TRANSCODE=False 입니다. 위 명령을 확인한 뒤 True로 바꾸고 다시 실행하세요.")
else:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for source, out_path, cmd in planned:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        print("변환 시작:", source.name)
        print("출력:", out_path)
        subprocess.run(cmd, check=True)
        print("완료:", out_path)


## 7. 변환 결과 검증

아래 셀은 출력 파일이 실제로 `H.264`, `1280x720`, `10fps`에 가까운지 확인합니다.

`RUN_TRANSCODE=True`로 변환을 실행한 뒤 이 셀을 실행하세요.


In [ ]:
result_rows = []
for source, out_path, _ in planned:
    row = {
        "source": video_summary(source),
        "output": video_summary(out_path) if out_path.exists() else None,
    }
    result_rows.append(row)

for row in result_rows:
    src = row["source"]
    out = row["output"]
    print("-" * 80)
    print("원본:", src["name"])
    print("  codec/fps/size:", src["codec"], src["fps"], f"{src['width']}x{src['height']}", f"{src['size_mb']}MB")
    if out is None:
        print("  출력 파일 없음")
        continue
    ratio = out["size_mb"] / src["size_mb"] if src["size_mb"] else 0
    print("출력:", out["name"])
    print("  codec/fps/size:", out["codec"], out["fps"], f"{out['width']}x{out['height']}", f"{out['size_mb']}MB")
    print("  bitrate:", f"{src['bitrate_kbps']}kbps -> {out['bitrate_kbps']}kbps")
    print("  용량비:", round(ratio, 3))

result_rows


## 8. 샘플 프레임 추출해서 눈으로 확인

아래 셀은 변환된 영상의 0초, 10초, 20초 지점 프레임을 jpg로 저장합니다.

확인할 것:

1. 좌측 상단 OSD 시간이 잘 보이는지
2. letterbox/padding이 의도대로 들어갔는지
3. 사람이 보기에도 모델 입력 품질이 너무 낮아지지 않았는지
4. 실제 live stream 화면과 구도/해상도 느낌이 비슷한지


In [ ]:
# 시작/중간/끝쪽 OSD timestamp 확인용 샘플입니다.
SAMPLE_TIMES_SEC = [0, 300, 540, 600]
FRAME_OUTPUT_DIR = OUTPUT_ROOT / "_sample_frames"

def extract_sample_frames(video_path: Path, output_dir: Path, times_sec: list[int]) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs = []
    for t in times_sec:
        out_jpg = output_dir / f"{video_path.stem}__t{t:04d}.jpg"
        cmd = [
            "ffmpeg", "-y",
            "-ss", str(t),
            "-i", str(video_path),
            "-frames:v", "1",
            str(out_jpg),
        ]
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        outputs.append(out_jpg)
    return outputs

sample_frame_paths = []
for _, out_path, _ in planned:
    if out_path.exists():
        sample_frame_paths.extend(extract_sample_frames(out_path, FRAME_OUTPUT_DIR, SAMPLE_TIMES_SEC))

print("저장된 샘플 프레임:")
for p in sample_frame_paths:
    print(p)


## 9. 자주 바꾸는 값 정리

### 용량이 너무 클 때

1. `CRF = 26`으로 변경
2. 그래도 크면 `CRF = 28`로 변경
3. live 유사성보다 용량이 중요하면 `LOW_LATENCY_MODE = False`로 변경

### 품질이 너무 낮을 때

1. `CRF = 20` 또는 `CRF = 22`로 변경
2. `PRESET = "medium"`으로 변경하면 같은 CRF에서 압축 효율이 좋아질 수 있지만 더 느립니다.

### live 입력과 더 가깝게 하고 싶을 때

1. `LOW_LATENCY_MODE = True` 유지
2. `GOP = TARGET_FPS * 2` 유지
3. 오디오 제거 유지: `DROP_AUDIO = True`

### 전체 폴더를 처리할 때

1. `MAX_VIDEOS_FOR_TEST = None`으로 변경
2. `RUN_TRANSCODE = True`로 변경
3. 1시간짜리 영상 여러 개면 시간이 오래 걸릴 수 있으므로 먼저 디스크 여유 공간을 확인하세요.


## 10. 카메라별 시간 정렬 trim

이 섹션은 이미 live 전처리가 끝난 720p/10fps 영상 3개를 서로 같은 시간축으로 맞추기 위한 전용 단계입니다.

현재 확인값 기준으로 `#1`, `#3`은 13:30:00 시작이고 `#2`는 13:29:00 시작이므로, `#2` 앞부분 60초를 잘라서 맞춥니다.

실행 순서:

1. 아래 `SYNC_INPUT_ROOT`가 맞는지 확인합니다.
2. `CAM_START_OFFSETS`에서 카메라별 자를 시간을 확인합니다.
3. 계획 셀을 실행해서 `trim_start_sec`, `output_duration_sec`를 확인합니다.
4. 문제가 없으면 `RUN_SYNC_TRIM = True`로 바꾼 뒤 실행 셀을 실행합니다.
5. 결과는 `_synced_by_time` 폴더에 저장되고, `sync_report.json`도 함께 저장됩니다.

주의: 이 단계는 원본을 수정하지 않습니다. 새 mp4 파일만 생성합니다.

노트북을 처음 열었다면 최소한 1~4번 셀을 먼저 실행해 `Path`, `video_summary`, `collect_videos` helper가 정의되게 하세요. 기존 live 변환 실행 셀은 건너뛰어도 됩니다.


In [ ]:
# 시간 동기화 대상 폴더입니다. 필요하면 이 경로만 바꿔서 재사용하세요.
SYNC_INPUT_ROOT = OUTPUT_ROOT

# 결과 저장 폴더입니다. 입력 폴더 아래에 새 폴더를 만들어 원본을 보존합니다.
SYNC_OUTPUT_ROOT = SYNC_INPUT_ROOT / "_synced_by_time"

# 10fps 기준 frame 단위 보정도 가능합니다.
# 예: 10fps에서 3프레임 늦추고 싶으면 {"sec": 0, "frames": 3} = 0.3초 trim입니다.
# 이번 260507 샘플은 별도 시작 offset을 적용하지 않습니다.
CAM_START_OFFSETS = {
    "#1": {"sec": 0, "frames": 0},
    "#2": {"sec": 0, "frames": 0},
    "#3": {"sec": 0, "frames": 0},
}

# None이면 하위 폴더까지 재귀 탐색, 0이면 현재 폴더 바로 아래 파일만 처리합니다.
SYNC_SCAN_DEPTH = 0

# 먼저 3개 영상 전체를 대상으로 계획만 확인합니다. 필요 시 1로 바꿔 단일 파일 테스트도 가능합니다.
SYNC_MAX_VIDEOS = None

# 실제 변환 실행 스위치입니다. 처음에는 반드시 False로 계획만 확인하세요.
RUN_SYNC_TRIM = False

print("SYNC_INPUT_ROOT  =", SYNC_INPUT_ROOT)
print("SYNC_OUTPUT_ROOT =", SYNC_OUTPUT_ROOT)
print("CAM_START_OFFSETS =", CAM_START_OFFSETS)


In [ ]:
def sync_camera_key(path: Path) -> str:
    """파일명에 포함된 '#1', '#2', '#3' 표기로 카메라를 구분합니다."""
    for key in CAM_START_OFFSETS:
        if key in path.name:
            return key
    raise ValueError(f"파일명에서 카메라 키를 찾지 못했습니다: {path.name}")


def sync_offset_seconds(camera_key: str, fps: float) -> float:
    offset = CAM_START_OFFSETS[camera_key]
    return float(offset.get("sec", 0)) + float(offset.get("frames", 0)) / float(fps)


def sync_collect_videos(input_root: Path, scan_depth: int | None = 0) -> list[Path]:
    return collect_videos(input_root, scan_depth)


def sync_output_path_for(source_path: Path) -> Path:
    camera_key = sync_camera_key(source_path).replace("#", "cam")
    return SYNC_OUTPUT_ROOT / f"{camera_key}_{source_path.stem}__synced.mp4"


sync_videos = sync_collect_videos(SYNC_INPUT_ROOT, SYNC_SCAN_DEPTH)
if SYNC_MAX_VIDEOS is not None:
    sync_videos = sync_videos[:SYNC_MAX_VIDEOS]

if not sync_videos:
    raise RuntimeError(f"동기화 대상 영상이 없습니다: {SYNC_INPUT_ROOT}")

sync_source_rows = []
for source in sync_videos:
    summary = video_summary(source)
    camera_key = sync_camera_key(source)
    trim_start = sync_offset_seconds(camera_key, summary["fps"] or TARGET_FPS)
    remain = summary["duration_sec"] - trim_start
    if remain <= 0:
        raise ValueError(f"trim 후 남는 길이가 없습니다: {source.name}, remain={remain}")
    sync_source_rows.append({
        "source": source,
        "summary": summary,
        "camera_key": camera_key,
        "trim_start_sec": round(trim_start, 3),
        "remaining_sec": round(remain, 3),
    })

# 끝나는 시간을 맞추기 위해 trim 후 남은 길이 중 가장 짧은 값을 공통 출력 길이로 사용합니다.
COMMON_OUTPUT_DURATION_SEC = round(min(row["remaining_sec"] for row in sync_source_rows), 3)

sync_plan = []
for row in sync_source_rows:
    output_path = sync_output_path_for(row["source"])
    sync_plan.append({
        **row,
        "output_path": output_path,
        "output_duration_sec": COMMON_OUTPUT_DURATION_SEC,
    })

print(f"동기화 대상 영상 수: {len(sync_plan)}")
print(f"공통 출력 길이: {COMMON_OUTPUT_DURATION_SEC} sec")
for item in sync_plan:
    print("-", item["camera_key"], item["source"].name)
    print("  fps/duration:", item["summary"]["fps"], item["summary"]["duration_sec"])
    print("  trim_start_sec:", item["trim_start_sec"])
    print("  output_duration_sec:", item["output_duration_sec"])
    print("  output:", item["output_path"])

sync_plan


In [ ]:
def build_sync_trim_cmd(item: dict) -> list[str]:
    # -ss를 -i 뒤에 두면 느릴 수 있지만, seek 정확도가 더 안정적입니다.
    cmd = [
        "ffmpeg", "-y",
        "-i", str(item["source"]),
        "-ss", str(item["trim_start_sec"]),
        "-t", str(item["output_duration_sec"]),
        "-map", "0:v:0",
        "-an", "-sn", "-dn",
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        "-g", str(GOP),
        "-keyint_min", str(GOP),
        "-sc_threshold", "0",
        "-movflags", "+faststart",
    ]
    if LOW_LATENCY_MODE:
        cmd += ["-tune", "zerolatency", "-bf", "0"]
    cmd += [str(item["output_path"])]
    return cmd


for item in sync_plan:
    item["cmd"] = build_sync_trim_cmd(item)
    print(" ".join(shlex.quote(part) for part in item["cmd"]))

if not RUN_SYNC_TRIM:
    print("RUN_SYNC_TRIM=False 입니다. 위 계획과 명령을 확인한 뒤 True로 바꾸고 다시 실행하세요.")
else:
    SYNC_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for item in sync_plan:
        item["output_path"].parent.mkdir(parents=True, exist_ok=True)
        print("동기화 변환 시작:", item["source"].name)
        subprocess.run(item["cmd"], check=True)
        print("완료:", item["output_path"])

    report = []
    for item in sync_plan:
        report.append({
            "camera_key": item["camera_key"],
            "source": str(item["source"]),
            "output": str(item["output_path"]),
            "trim_start_sec": item["trim_start_sec"],
            "output_duration_sec": item["output_duration_sec"],
            "source_summary": item["summary"],
            "output_summary": video_summary(item["output_path"]),
        })

    report_path = SYNC_OUTPUT_ROOT / "sync_report.json"
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("report 저장:", report_path)
    report


### 동기화 결과 확인 방법

실행 후 `_synced_by_time` 폴더에서 세 mp4를 같은 시점으로 열어 좌측 상단 timestamp가 맞는지 확인하세요.

노트북에서 바로 확인하려면 아래 셀로 0초, 10초, 30초 샘플 프레임을 추출한 뒤 jpg를 열어 비교하면 됩니다.

만약 `#2`가 여전히 빠르면 `#2`의 `sec` 또는 `frames`를 늘리고, 느리면 줄인 뒤 다시 실행하세요.


In [ ]:
SYNC_SAMPLE_TIMES_SEC = [0, 10, 30]
SYNC_FRAME_OUTPUT_DIR = SYNC_OUTPUT_ROOT / "_sample_frames"

sync_sample_frame_paths = []
for item in sync_plan:
    output_path = item["output_path"]
    if output_path.exists():
        sync_sample_frame_paths.extend(extract_sample_frames(output_path, SYNC_FRAME_OUTPUT_DIR, SYNC_SAMPLE_TIMES_SEC))

print("저장된 동기화 샘플 프레임:")
for path in sync_sample_frame_paths:
    print(path)


## 11. 준비된 output 영상에서 동일 구간 clip 추출

이미 live 전처리가 끝난 output 영상 3개가 있으면, 원본 전처리 cell을 다시 실행하지 않고 이 section만 사용할 수 있습니다.

### 노트북 재시작 후, 준비된 영상만 clip으로 자르는 실행 순서

커널을 재시작한 상태라면 아래 2개 code cell만 순서대로 실행하세요.

1. `025` clip 대상 폴더, 구간, helper 준비
2. `026` clip 추출 실행

즉, 이미 준비된 output 영상만 자를 때는 `002~023`의 전처리/동기화 cell을 실행하지 않아도 됩니다.

단, 편집 세팅 자체를 바꿨거나 새 output을 다시 만들어야 하는 경우에는 위쪽 전처리 cell을 먼저 실행한 뒤 이 section을 실행하세요.

이번 기본 목적은 아래 폴더의 준비된 3개 영상을 모두 `8분 00초 ~ 9분 10초` 구간으로 추출하는 것입니다.

```text
/share_ssd/ltb/Users/ltb/git_repos/video-preprocess-workbench/artifacts/runs/20260507_155115_260507_샘플다시저장_원본영상_sample_live_720p_10fps_aligned
```

clip 단계는 이미 편집된 output을 입력으로 받으므로 `flip`, `resize`, `pad`, `fps 변환`을 다시 적용하지 않습니다. 대신 frame 기준 `trim` 후 기존 live 인코딩 preset으로 재인코딩해서 3개 영상의 같은 초 구간을 안정적으로 맞춥니다.


In [ ]:
# 이미 준비된 live output 3개에서 동일 초 구간 clip만 추출하기 위한 설정입니다.
# 커널 재시작 후에도 이 cell과 다음 cell만 실행하면 동작하도록 필요한 helper를 함께 정의합니다.
import json
import shlex
import subprocess
from pathlib import Path

# 위쪽 live preset cell을 실행하지 않은 경우를 위한 기본값입니다.
TARGET_FPS = globals().get("TARGET_FPS", 10)
GOP = globals().get("GOP", TARGET_FPS * 2)
CRF = globals().get("CRF", 23)
PRESET = globals().get("PRESET", "veryfast")
LOW_LATENCY_MODE = globals().get("LOW_LATENCY_MODE", True)
VIDEO_EXTS = globals().get("VIDEO_EXTS", {".mp4", ".mov", ".mkv", ".avi"})


def collect_videos(input_root: Path, scan_depth: int | None = 0) -> list[Path]:
    input_root = input_root.expanduser().resolve()
    if input_root.is_file():
        return [input_root] if input_root.suffix.lower() in VIDEO_EXTS else []
    if not input_root.exists():
        raise FileNotFoundError(input_root)
    if scan_depth is None:
        files = [p for p in input_root.rglob("*") if p.suffix.lower() in VIDEO_EXTS]
    else:
        files = []
        for p in input_root.glob("*"):
            if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
                files.append(p)
    return sorted(files)


def run_cmd(cmd: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, check=True, capture_output=True, text=True)


def ffprobe_json(path: Path) -> dict:
    cmd = [
        "ffprobe", "-v", "error",
        "-print_format", "json",
        "-show_format", "-show_streams",
        str(path),
    ]
    return json.loads(run_cmd(cmd).stdout)


def parse_rate(value: str | None) -> float:
    if not value or value == "0/0":
        return 0.0
    if "/" in value:
        n, d = value.split("/", 1)
        return float(n) / float(d) if float(d) else 0.0
    return float(value)


def video_summary(path: Path) -> dict:
    data = ffprobe_json(path)
    stream = next(s for s in data["streams"] if s.get("codec_type") == "video")
    fmt = data["format"]
    size_bytes = int(fmt.get("size", 0) or 0)
    duration = float(stream.get("duration") or fmt.get("duration") or 0.0)
    bitrate = float(stream.get("bit_rate") or fmt.get("bit_rate") or 0.0)
    return {
        "path": str(path),
        "name": path.name,
        "codec": stream.get("codec_name"),
        "width": int(stream.get("width") or 0),
        "height": int(stream.get("height") or 0),
        "fps": round(parse_rate(stream.get("avg_frame_rate")), 3),
        "duration_sec": round(duration, 3),
        "bitrate_kbps": round(bitrate / 1000, 1) if bitrate else 0.0,
        "size_mb": round(size_bytes / 1024 / 1024, 2),
    }


CLIP_INPUT_ROOT = Path("/share_ssd/ltb/Users/ltb/git_repos/video-preprocess-workbench/artifacts/runs/20260507_155115_260507_샘플다시저장_원본영상_sample_live_720p_10fps_aligned")

# 8분 00초 ~ 9분 10초 = 480초 ~ 550초
CLIP_START_SEC = 8 * 60
CLIP_END_SEC = 9 * 60 + 10

# 준비된 output 폴더 바로 아래 mp4 3개만 대상으로 봅니다.
CLIP_SCAN_DEPTH = 0
EXPECTED_CLIP_VIDEO_COUNT = 3

# 결과는 입력 폴더 안의 새 하위 폴더에 저장해서 기존 output을 보존합니다.
CLIP_OUTPUT_ROOT = CLIP_INPUT_ROOT / "_clips" / f"clip_{CLIP_START_SEC:04d}-{CLIP_END_SEC:04d}s"

# 실제 clip 생성 스위치입니다. 계획만 확인하려면 False로 바꾸세요.
RUN_CLIP_EXPORT = True

clip_videos = collect_videos(CLIP_INPUT_ROOT, CLIP_SCAN_DEPTH)
if EXPECTED_CLIP_VIDEO_COUNT is not None and len(clip_videos) != EXPECTED_CLIP_VIDEO_COUNT:
    raise RuntimeError(f"clip 대상 영상 수가 예상과 다릅니다: expected={EXPECTED_CLIP_VIDEO_COUNT}, actual={len(clip_videos)}")
if CLIP_END_SEC <= CLIP_START_SEC:
    raise ValueError("CLIP_END_SEC는 CLIP_START_SEC보다 커야 합니다.")

print("CLIP_INPUT_ROOT  =", CLIP_INPUT_ROOT)
print("CLIP_OUTPUT_ROOT =", CLIP_OUTPUT_ROOT)
print("CLIP_RANGE_SEC   =", CLIP_START_SEC, "~", CLIP_END_SEC)
print("ENCODING_PRESET  =", {"TARGET_FPS": TARGET_FPS, "GOP": GOP, "CRF": CRF, "PRESET": PRESET, "LOW_LATENCY_MODE": LOW_LATENCY_MODE})
print(f"clip 대상 영상 수: {len(clip_videos)}")
for idx, path in enumerate(clip_videos):
    summary = video_summary(path)
    print(f"[{idx:03d}] {path.name}")
    print("     fps/duration/size:", summary["fps"], summary["duration_sec"], f"{summary['width']}x{summary['height']}")


In [ ]:
def clip_output_path_for(source_path: Path) -> Path:
    return CLIP_OUTPUT_ROOT / f"{source_path.stem}__clip_{CLIP_START_SEC:04d}-{CLIP_END_SEC:04d}s.mp4"


def build_frame_clip_cmd(source_path: Path, output_path: Path) -> tuple[list[str], dict]:
    summary = video_summary(source_path)
    fps = float(summary["fps"] or TARGET_FPS)
    start_frame = round(CLIP_START_SEC * fps)
    end_frame = round(CLIP_END_SEC * fps)
    output_frames = end_frame - start_frame
    if output_frames <= 0:
        raise ValueError(f"clip frame 범위가 잘못됐습니다: {source_path.name}")
    if summary["duration_sec"] < CLIP_END_SEC:
        raise ValueError(f"영상 길이가 clip 종료 초보다 짧습니다: {source_path.name}, duration={summary['duration_sec']}")

    vf = f"trim=start_frame={start_frame}:end_frame={end_frame},setpts=PTS-STARTPTS"
    cmd = [
        "ffmpeg", "-y",
        "-i", str(source_path),
        "-map", "0:v:0",
        "-an", "-sn", "-dn",
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),
        "-pix_fmt", "yuv420p",
        "-g", str(GOP),
        "-keyint_min", str(GOP),
        "-sc_threshold", "0",
        "-movflags", "+faststart",
        "-r", f"{fps:g}",
    ]
    if LOW_LATENCY_MODE:
        cmd += ["-tune", "zerolatency", "-bf", "0"]
    cmd += [str(output_path)]
    meta = {
        "fps": fps,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "output_frames": output_frames,
        "expected_duration_sec": output_frames / fps,
    }
    return cmd, meta


clip_plan = []
for source in clip_videos:
    output_path = clip_output_path_for(source)
    cmd, meta = build_frame_clip_cmd(source, output_path)
    clip_plan.append({
        "source": source,
        "output_path": output_path,
        "cmd": cmd,
        "meta": meta,
    })

for item in clip_plan:
    print("SOURCE:", item["source"].name)
    print("OUTPUT:", item["output_path"])
    print("META:", item["meta"])
    print("CMD:", " ".join(shlex.quote(part) for part in item["cmd"]))
    print()

if not RUN_CLIP_EXPORT:
    print("RUN_CLIP_EXPORT=False 입니다. 위 계획을 확인한 뒤 True로 바꾸고 다시 실행하세요.")
else:
    CLIP_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    clip_report = []
    for item in clip_plan:
        print("clip 추출 시작:", item["source"].name)
        subprocess.run(item["cmd"], check=True)
        output_summary = video_summary(item["output_path"])
        clip_report.append({
            "source": str(item["source"]),
            "output": str(item["output_path"]),
            "clip_start_sec": CLIP_START_SEC,
            "clip_end_sec": CLIP_END_SEC,
            "frame_meta": item["meta"],
            "output_summary": output_summary,
        })
        print("완료:", item["output_path"])
        print("  output fps/duration/frames:", output_summary["fps"], output_summary["duration_sec"], round(output_summary["duration_sec"] * output_summary["fps"]))

    report_path = CLIP_OUTPUT_ROOT / "clip_report.json"
    report_path.write_text(json.dumps(clip_report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("clip report 저장:", report_path)
    clip_report
